### 3.9.27. First-Order Methods

$$
\mathbf{x}_{k+1} = \operatorname{prox}_{t\,h}\!\big(\mathbf{x}_k - t\,\nabla g(\mathbf{x}_k)\big),
\qquad
\operatorname{prox}_{t h}(\mathbf{v}) = \arg\min_{\mathbf{u}} \Big(h(\mathbf{u}) + \tfrac{1}{2t}\|\mathbf{u}-\mathbf{v}\|^2\Big).
$$

**Explanation:**

First-order methods use only gradients (or subgradients), never Hessians, which makes each iteration cheap and scalable to very large problems. For a composite objective $f = g + h$ with $g$ smooth and $h$ convex but possibly nonsmooth, the *proximal gradient* method alternates a gradient step on $g$ with the proximal operator of $h$. When $h(\mathbf{x}) = \lambda\|\mathbf{x}\|_1$ the proximal operator is the closed-form soft-thresholding map, which is why first-order methods dominate sparse regression and large-scale [convex problems](../03_Convex_Optimization/29_convex_problem_standard_form_and_optimality.ipynb). They contrast with the [Newton methods](./11_newtons_method.ipynb) that use curvature for faster local convergence.

**Numerical Example:**

Minimize the one-dimensional lasso objective
$$
f(x) = \tfrac12 (x - 3)^2 + \lambda|x|,\qquad \lambda = 1.
$$

The smooth part has gradient $g'(x) = x - 3$, and the proximal operator of $\lambda|x|$ is soft-thresholding $S_\lambda(v) = \operatorname{sign}(v)\max(|v|-\lambda, 0)$. A proximal-gradient step with $t=1$ from $x_0 = 0$ gives
$$
v = x_0 - t\,g'(x_0) = 0 - (0 - 3) = 3,\qquad
x_1 = S_1(3) = 2.
$$
The fixed point is $x^\star = S_1(x^\star - (x^\star - 3)) = S_1(3) = 2$, matching the exact minimizer of $f$.

In [1]:
import sympy as sp

x, v, t, lam = sp.symbols("x v t lambda", real=True, positive=True)
proximal_argument = lam*sp.Abs(x) + (x - v)**2 / (2*t)
soft_threshold = sp.Piecewise((v - lam, v > lam), (v + lam, v < -lam), (0, True))

print("prox of lambda|x| (soft-threshold) ="); sp.pprint(soft_threshold)

smooth_gradient = lambda point: point - 3
iterate = 0.0
for step in range(6):
    gradient_point = iterate - 1.0*smooth_gradient(iterate)
    iterate = float(soft_threshold.subs({v: gradient_point, lam: 1}))
    print(f"iter {step+1}: x = {iterate}")

prox of lambda|x| (soft-threshold) =
⎧-λ + v  for λ < v
⎨                 
⎩  0     otherwise
iter 1: x = 2.0
iter 2: x = 2.0
iter 3: x = 2.0
iter 4: x = 2.0
iter 5: x = 2.0
iter 6: x = 2.0


**Equivalent numpy implementation:**

In [2]:
import numpy as np

def soft_threshold(value, penalty):
    return np.sign(value) * max(abs(value) - penalty, 0.0)

penalty, step_size = 1.0, 1.0
iterate = 0.0
for _ in range(6):
    gradient_step = iterate - step_size * (iterate - 3.0)
    iterate = soft_threshold(gradient_step, step_size * penalty)

print("proximal-gradient minimizer x =", iterate)

proximal-gradient minimizer x = 2.0


**References:**

[📘 Beck, A. (2017). *First-Order Methods in Optimization*. SIAM.](https://doi.org/10.1137/1.9781611974997)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Preconditioned Conjugate Gradient and Nonquadratic Use](./26_preconditioned_conjugate_gradient_and_nonquadratic_use.ipynb) | [Next: Incremental Gradient Methods ➡️](./28_incremental_gradient_methods.ipynb)